# N2 · 搭一个 mini-LLaVA (投影路线)

> 配套 10.2-L3 · 用投影连接器把 10.1 的 mini-ViT 真接到一个 tiny LLM, 跑通「图+问题→输出」前向,
> 亲手搭最小 LLaVA。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
VIS_SRC = Path.cwd().parent.parent / "vision-encoders" / "src"  # 复用 10.1 mini-ViT
for p in (SRC, VIS_SRC): sys.path.insert(0, str(p))
import torch, torch.nn as nn
import tiny_vit as tv
import connectors as cn
print('就绪 (复用 10.1 mini-ViT)')

就绪 (复用 10.1 mini-ViT)


## 1. 视觉一半: 用 10.1 的 mini-ViT 把图变视觉 token

In [2]:
torch.manual_seed(0)
img = tv.make_synthetic_image('blocks', size=16, seed=2)
patches = tv.patchify(img, patch=4)
vit = tv.build_tiny_vit(patch_dim=patches.shape[1], n_patches=patches.shape[0], d_model=32)
vis_tokens = vit(torch.tensor(patches[None], dtype=torch.float32))[:, 1:]  # 取 patch token (去 CLS)
print(f"视觉 token: {tuple(vis_tokens.shape)} (16 个 patch token, 32 维)")

视觉 token: (1, 16, 32) (16 个 patch token, 32 维)


## 2. 投影连接器: 把视觉 token 投到 LLM embedding 空间 (L3 的 MLP)

In [3]:
LLM_DIM = 48
proj = cn.build_connectors(vis_dim=32, llm_dim=LLM_DIM, n_vis=16)['projection']
# 模拟文本 token (一个 tiny 文本 embedding: '这张图是什么')
vocab = 50; txt_ids = torch.tensor([[5, 12, 7, 3, 9]])
txt_embed = nn.Embedding(vocab, LLM_DIM)
txt_tokens = txt_embed(txt_ids)
fused = proj(vis_tokens, txt_tokens)   # [视觉投影 ×16 | 文本 ×5]
print(f"融合输入序列: {tuple(fused.shape)} = (batch, 16 视觉 + 5 文本 = 21, {LLM_DIM})")

融合输入序列: (1, 21, 48) = (batch, 16 视觉 + 5 文本 = 21, 48)


## 3. LLM 一半: 一个 tiny transformer 处理融合序列, 输出下一 token 分布

In [4]:
torch.manual_seed(0)
llm = nn.TransformerEncoder(nn.TransformerEncoderLayer(LLM_DIM, 4, LLM_DIM*2, batch_first=True), 2)
head = nn.Linear(LLM_DIM, vocab)
out = llm(fused)              # 视觉和文本 token 互相 attend (统一自注意力)
logits = head(out[:, -1])     # 最后位置预测下一 token
print(f"输出 logits: {tuple(logits.shape)} (词表 {vocab} 上的分布)")
print(f"预测的下一 token id: {int(logits.argmax(-1))}")
print("\n→ 这就是一个最小 LLaVA: 图 → ViT → 投影 → 拼文本 → LLM → 输出。")

输出 logits: (1, 50) (词表 50 上的分布)
预测的下一 token id: 41

→ 这就是一个最小 LLaVA: 图 → ViT → 投影 → 拼文本 → LLM → 输出。


## 4. 反思 (10.2 收口)

你亲手搭了一个投影路线的 mini-LLaVA: **视觉塔(10.1) + MLP 投影(10.2) + tiny LLM**, 视觉和文本
token 在统一自注意力里互相 attend。带走:
- 三路线 (投影/cross-attn/early-fusion) 各优化什么、决策树怎么选 (N1)。
- 投影路线极简 (一个 MLP) 却 work, 因为视觉塔已懂语言 (10.1 对比学习)。
- 这个 mini-LLaVA 还没训 —— 它现在是随机权重。怎么真正训出能问答的 VLM?

> 交棒 10.3: `vlm-training-recipe` —— 两阶段配方 (对齐预训练 + 指令微调)、数据、冻结策略,
> 把这个随机 mini-LLaVA 真正训成能回答图像问题的 VLM。